In [ ]:
import os
import json
import hashlib
import getpass
import uuid
from datetime import datetime

class TaskManager:
    def __init__(self):
        self.current_user = None
        self.users_file = "users.json"
        self.tasks_dir = "user_tasks"
        
        # Create tasks directory if it doesn't exist
        if not os.path.exists(self.tasks_dir):
            os.makedirs(self.tasks_dir)
            
        # Create users file if it doesn't exist
        if not os.path.exists(self.users_file):
            with open(self.users_file, 'w') as f:
                json.dump({}, f)
    
    def hash_password(self, password):
            """Upgraded: Hash a password with a static salt."""
            salt = "s0me_r4ndom_s4lt" # In a real app, use a unique salt per user
            return hashlib.sha256((password + salt).encode()).hexdigest()
    
    def register(self):
        """Register a new user."""
        print("\n=== User Registration ===")
        
        # Load existing users
        with open(self.users_file, 'r') as f:
            users = json.load(f)
        
        # Get username
        while True:
            username = input("Enter username: ").strip()
            if not username:
                print("Username cannot be empty.")
                continue
            if username in users:
                print("Username already exists. Please choose another.")
                continue
            break
        
        # Get password
        while True:
            password = getpass.getpass("Enter password: ")
            if not password:
                print("Password cannot be empty.")
                continue
            confirm_password = getpass.getpass("Confirm password: ")
            if password != confirm_password:
                print("Passwords do not match. Try again.")
                continue
            break
        
        # Store user
        users[username] = {
            "password": self.hash_password(password),
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        with open(self.users_file, 'w') as f:
            json.dump(users, f)
        
        print(f"User '{username}' registered successfully!")
        return username
    
    def login(self):
        """Login a user."""
        print("\n=== User Login ===")
        
        # Load existing users
        with open(self.users_file, 'r') as f:
            users = json.load(f)
        
        # Get credentials
        username = input("Username: ").strip()
        password = getpass.getpass("Password: ")
        
        # Validate credentials
        if username in users and users[username]["password"] == self.hash_password(password):
            self.current_user = username
            print(f"Welcome back, {username}!")
            return True
        else:
            print("Invalid username or password.")
            return False
    
    def get_user_tasks_file(self):
        """Get the path to the current user's tasks file."""
        return os.path.join(self.tasks_dir, f"{self.current_user}_tasks.json")
    
    def load_tasks(self):
        """Load tasks for the current user."""
        tasks_file = self.get_user_tasks_file()
        
        if not os.path.exists(tasks_file):
            return {}
        
        with open(tasks_file, 'r') as f:
            return json.load(f)
    
    def save_tasks(self, tasks):
        """Save tasks for the current user."""
        tasks_file = self.get_user_tasks_file()
        
        with open(tasks_file, 'w') as f:
            json.dump(tasks, f, indent=2)
    
    def add_task(self):
        """Add a new task for the current user."""
        if not self.current_user:
            print("You must be logged in to add tasks.")
            return
        
        print("\n=== Add New Task ===")
        description = input("Enter task description: ").strip()
        
        if not description:
            print("Task description cannot be empty.")
            return
        
        # Generate a unique task ID
        task_id = str(uuid.uuid4())[:8]
        
        # Create task object
        task = {
            "id": task_id,
            "description": description,
            "status": "Pending",
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        # Load existing tasks
        tasks = self.load_tasks()
        
        # Add new task
        tasks[task_id] = task
        
        # Save tasks
        self.save_tasks(tasks)
        
        print(f"Task added successfully! Task ID: {task_id}")
    
    def view_tasks(self):
        """View all tasks for the current user."""
        if not self.current_user:
            print("You must be logged in to view tasks.")
            return
        
        tasks = self.load_tasks()
        
        if not tasks:
            print("\nYou have no tasks.")
            return
        
        print(f"\n=== Tasks for {self.current_user} ===")
        print(f"{'ID':<10} {'Status':<12} {'Description':<50}")
        print("-" * 72)
        
        for task_id, task in tasks.items():
            print(f"{task_id:<10} {task['status']:<12} {task['description']:<50}")
    
    def mark_completed(self):
        """Mark a task as completed."""
        if not self.current_user:
            print("You must be logged in to update tasks.")
            return
        
        tasks = self.load_tasks()
        
        if not tasks:
            print("\nYou have no tasks to update.")
            return
        
        print("\n=== Mark Task as Completed ===")
        task_id = input("Enter the ID of the task to mark as completed: ").strip()
        
        if task_id in tasks:
            tasks[task_id]["status"] = "Completed"
            self.save_tasks(tasks)
            print(f"Task {task_id} marked as completed!")
        else:
            print(f"Task with ID {task_id} not found.")
    
    def delete_task(self):
        """Delete a task."""
        if not self.current_user:
            print("You must be logged in to delete tasks.")
            return
        
        tasks = self.load_tasks()
        
        if not tasks:
            print("\nYou have no tasks to delete.")
            return
        
        print("\n=== Delete Task ===")
        task_id = input("Enter the ID of the task to delete: ").strip()
        
        if task_id in tasks:
            del tasks[task_id]
            self.save_tasks(tasks)
            print(f"Task {task_id} deleted successfully!")
        else:
            print(f"Task with ID {task_id} not found.")
    
    def logout(self):
        """Logout the current user."""
        if self.current_user:
            print(f"Goodbye, {self.current_user}!")
            self.current_user = None
        else:
            print("No user is currently logged in.")

def main():
    manager = TaskManager()
    
    while True:
        if not manager.current_user:
            print("\n=== Task Manager ===")
            print("1. Register\n2. Login\n3. Exit")
            choice = input("Choice: ")
            if choice == '1': manager.register()
            elif choice == '2': manager.login()
            elif choice == '3': break
        else:
            print(f"\n--- Logged in: {manager.current_user} ---")
            print("1. Add Task\n2. View Tasks\n3. Complete Task\n4. Delete Task\n5. Logout\n6. Exit")
            choice = input("Choice: ")
            
            if choice == '1': manager.add_task()
            elif choice == '2': manager.view_tasks()
            elif choice == '3': manager.mark_completed()
            elif choice == '4': manager.delete_task()
            elif choice == '5': manager.logout()
            elif choice == '6': break

if __name__ == "__main__":
    main()


=== Task Manager ===
1. Register
2. Login
3. Exit


Choice:  2



=== User Login ===


Username:  sush
Password:  ········


Welcome back, sush!

--- Logged in: sush ---
1. Add Task
2. View Tasks
3. Complete Task
4. Delete Task
5. Logout
6. Exit
